# Limpieza y Graficacion

# Tipo de limpieza:
Eliminación de datos irrelevantes (Filtrado): Separa las variables o filas que no aportan valor a la pregunta de negocio.

Corrección de tipos de datos: Transforma cadenas de texto a formatos correctos.Estandarización de formatos: 

Unifica criterios para evitar que variaciones en la escritura afecten el análisis.

# Análisis de producción eléctrica en América Latina

Este notebook prepara una vista clara y útil para la toma de decisiones sobre la evolución de la producción eléctrica por país.

## Objetivo
- Cargar el conjunto de datos global
- Filtrar los países de América Latina
- Calcular indicadores clave por país
- Generar una gráfica comparativa visual

> La idea es transformar los datos crudos en una historia fácil de interpretar para análisis estratégico.

# Script de Preprocesamiento: Segmentación Temporal y Geográfica de Producción Eléctrica en LATAM

In [1]:
import os
import pandas as pd

INPUT_FILE = 'global_electricity_production_data.csv'
OUTPUT_FILE = 'latam_electricity_production_data.csv'

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f'No existe el archivo: {INPUT_FILE}')

# Cargar datos
df = pd.read_csv(INPUT_FILE)
print('Archivo cargado:', INPUT_FILE)
print('Columnas disponibles:', list(df.columns))

# Detectar columna de fecha y convertirla a datetime (si existe)
date_col = next((c for c in df.columns if 'date' in c.lower()), None)
if date_col:
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
else:
    raise ValueError('No se encontró una columna de fecha en el dataset.')

# Detectar columna de país
country_col = next((c for c in df.columns if 'country' in c.lower() or 'pais' in c.lower()), None)
if country_col is None:
    raise ValueError(f'No se encontró la columna de país. Columnas disponibles: {list(df.columns)}')

paises_latam = [
    'Argentina', 'Bolivia', 'Brazil', 'Brasil', 'Chile', 'Colombia',
    'Costa Rica', 'Cuba', 'Dominican Republic', 'República Dominicana',
    'Ecuador', 'El Salvador', 'Guatemala', 'Honduras', 'Mexico', 'México',
    'Nicaragua', 'Panama', 'Panamá', 'Paraguay', 'Peru', 'Perú',
    'Puerto Rico', 'Uruguay', 'Venezuela'
]

# Aplicar filtro por país y por fecha (desde 2016-01-01 en adelante)
cutoff_date = pd.to_datetime('2016-01-01')
df_latam = df[ df[country_col].isin(paises_latam) & (df[date_col] >= cutoff_date) ].copy()

if df_latam.empty:
    raise ValueError('No se encontraron registros de América Latina en el conjunto de datos después del filtro de fecha.')

# Guardar resultado limpiado
df_latam.to_csv(OUTPUT_FILE, index=False)
print('Archivo limpio generado:', OUTPUT_FILE)
print('Registros finales:', len(df_latam))
print('\nVista previa de los datos filtrados:')
print(df_latam.head())

Archivo cargado: global_electricity_production_data.csv
Columnas disponibles: ['country_name', 'date', 'parameter', 'product', 'value', 'unit']
Archivo limpio generado: latam_electricity_production_data.csv
Registros finales: 8846

Vista previa de los datos filtrados:
   country_name       date                   parameter  \
67        Chile 2023-12-01                     Remarks   
68        Chile 2023-12-01  Net Electricity Production   
69        Chile 2023-12-01  Net Electricity Production   
70        Chile 2023-12-01  Net Electricity Production   
71        Chile 2023-12-01  Net Electricity Production   

                              product      value unit  
67   Data is estimated for this month        NaN  GWh  
68                        Electricity  7734.9912  GWh  
69            Total Combustible Fuels  2099.8702  GWh  
70  Coal, Peat and Manufactured Gases  1077.3487  GWh  
71         Oil and Petroleum Products    24.9683  GWh  


# Resumen Agregado: Estadísticas Descriptivas de Generación Eléctrica en América Latina

In [2]:
import pandas as pd

df = pd.read_csv('latam_electricity_production_data.csv')
print(df.head())
print('\nColumnas disponibles:', list(df.columns))

if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])

country_col = next((c for c in df.columns if 'country' in c.lower() or 'pais' in c.lower()), None)
if country_col is None:
    raise ValueError('No se encontró la columna de país para el cálculo.')

value_col = 'value' if 'value' in df.columns else df.columns[-1]

resumen = (
    df.groupby(country_col)[value_col]
    .agg(['mean', 'max', 'min', 'sum'])
)
resumen.columns = ['promedio', 'maximo', 'minimo', 'total']
print('\nResumen por país:')
print(resumen.sort_values('total', ascending=False).head(10))

  country_name        date                   parameter  \
0        Chile  2023-12-01                     Remarks   
1        Chile  2023-12-01  Net Electricity Production   
2        Chile  2023-12-01  Net Electricity Production   
3        Chile  2023-12-01  Net Electricity Production   
4        Chile  2023-12-01  Net Electricity Production   

                             product      value unit  
0   Data is estimated for this month        NaN  GWh  
1                        Electricity  7734.9912  GWh  
2            Total Combustible Fuels  2099.8702  GWh  
3  Coal, Peat and Manufactured Gases  1077.3487  GWh  
4         Oil and Petroleum Products    24.9683  GWh  

Columnas disponibles: ['country_name', 'date', 'parameter', 'product', 'value', 'unit']

Resumen por pa?s:
                  promedio      maximo  minimo         total
country_name                                                
Brazil        11112.359287  61201.3215  0.0000  1.520171e+07
Mexico         5964.531514  39

# Graficacion de Datos Limpios

In [4]:
import pandas as pd
import plotly.express as px

# Cargar datos limpios
df = pd.read_csv('latam_electricity_production_data.csv')

# Detectar columna de país
country_col = next((c for c in df.columns if 'country' in c.lower() or 'pais' in c.lower()), None)
if country_col is None:
    raise ValueError('No se encontró columna de país.')

# Fecha
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])

# Columna de valores
value_col = 'value' if 'value' in df.columns else df.columns[-1]

# Preparar datos para gráfica
df_graph = df.groupby([country_col, 'date'], as_index=False)[value_col].sum()
df_graph = df_graph.sort_values(['date', country_col])

# Visual: tema oscuro con contraste
background_color = '#0f1724'  # fondo oscuro neutro
text_color = '#e6eef8'

fig = px.line(
    df_graph,
    x='date',
    y=value_col,
    color=country_col,
    title='Producción eléctrica por país en América Latina',
    labels={'date': 'Fecha', value_col: 'Producción (GWh)', country_col: 'País'},
    markers=False
)

fig.update_layout(
    template='plotly_dark',
    plot_bgcolor=background_color,
    paper_bgcolor=background_color,
    font=dict(color=text_color),
    title={'x': 0.5, 'xanchor': 'center', 'font': {'size': 20}},
    legend_title='País',
    legend=dict(bgcolor='rgba(0,0,0,0.4)', bordercolor='rgba(255,255,255,0.1)'),
    height=650,
    margin=dict(l=40, r=20, t=80, b=40)
)

fig.update_traces(line={'width': 2.5}, selector=dict(mode='lines'))
fig.update_xaxes(showgrid=True, gridcolor='rgba(255,255,255,0.05)', title_text='Fecha')
fig.update_yaxes(showgrid=True, gridcolor='rgba(255,255,255,0.05)', title_text='Producción (GWh)')

# Mostrar y exportar
fig.show()
fig.write_html('latam_produccion_electricidad.html', include_plotlyjs='cdn')
print('Gráfica exportada a latam_produccion_electricidad.html')

Gráfica exportada a latam_produccion_electricidad.html
